# Phase 1: Offline Distillation Data Generation (OpenAI Client Version)
Notebook này sinh dữ liệu ma trận khoảng cách $D$ giữa các rollouts giải toán.
**Được tối ưu hóa:** Sử dụng thư viện `openai` để gọi tới Hugging Face Router. Cực kỳ gọn nhẹ và ổn định.

In [ ]:
# [QUAN TRỌNG] Kết nối với Google Drive để đảm bảo lưu dữ liệu an toàn
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/Data_Phase1', exist_ok=True)
    print("Đã kết nối Google Drive thành công!")
except:
    print("Không chạy trên môi trường Colab.")

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q datasets jsonlines tqdm openai

## Cấu hình API Key
Nhập mã HF_TOKEN của bạn vào đây.

In [ ]:
import os
import phase1_distillation.config as config

# ĐIỀN API KEY CỦA BẠN VÀO ĐÂY (HÃY XOÁ TRƯỚC KHI PUSH GITHUB)
os.environ['HF_TOKEN'] = 'your_hf_token_here'

print(f"API Base URL: {config.API_BASE_URL}")
if os.getenv('HF_TOKEN') and os.getenv('HF_TOKEN') != 'your_hf_token_here':
    print("Token status: Đã thiết lập thành công.")

## Chạy Pipeline (Sinh K=4 Rollouts và Ma trận $D$)

In [ ]:
import jsonlines
from tqdm import tqdm
from phase1_distillation import MathDataset, MathRolloutGenerator, AlignmentJudge
from phase1_distillation.dataset import get_problem_id
import phase1_distillation.config as config

os.makedirs(os.path.dirname(config.DRIVE_OUTPUT_FILE), exist_ok=True)

# 1. Khởi tạo Dataset
dataset, processed_ids = MathDataset.load_with_resume(config.DRIVE_PROCESSED_IDS, sample_size=100)

# 2. Khởi tạo Generator & Judge
generator = MathRolloutGenerator(model_id=config.MODEL_ID)
judge = AlignmentJudge(model_id=config.MODEL_ID)

# 3. Vòng lặp chính
with jsonlines.open(config.DRIVE_OUTPUT_FILE, mode='a') as writer, open(config.DRIVE_PROCESSED_IDS, "a") as track_file:
    for item in tqdm(dataset, total=len(dataset), desc="Generating Distillation Data"):
        problem_id = get_problem_id(item['problem'])
        
        # Sinh 4 rollouts
        rollouts = generator.generate(item['problem'])
        
        # Sinh ma trận khoảng cách
        distance_matrices = judge.evaluate_pairs(item['problem'], rollouts)
        
        if distance_matrices is None or all(v is None for v in distance_matrices.values()):
            continue
            
        result = {
            "question_id": problem_id,
            "question": item['problem'],
            "ground_truth": item['ground_truth_solution'],
            "generated_rollouts": rollouts,
            "distance_matrices": distance_matrices
        }
        writer.write(result)
        
        track_file.write(problem_id + "\n")
        track_file.flush()
